In [1]:
import torch
import torchaudio
import librosa
import soundfile as sf
import random
import io
from pydub import AudioSegment
from pydub.generators import WhiteNoise

c:\Users\mxlov\anaconda3\envs\speech-recog-go\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
AUDIO_PATH = './speech.wav'
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

## Predictor / Transcriber

In [ ]:
def Transcriber (audiopath):
    # load
    waveform, sample_rate = torchaudio.load(audiopath)

    # resample
    if sample_rate != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

    # emission
    with torch.inference_mode():
        emission, _ = model(waveform)

    print(emission)

    # class
    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor):
            idx = torch.argmax(emission, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return " ".join([self.labels[i] for i in idx])

    # transcribe
    decoder = GreedyCTCDecoder(bundle.get_labels())
    txt = decoder(emission[0])
    txt = txt.replace("|", " ")
    return txt

## Support Function

In [4]:
def ExportMenu (audio: AudioSegment):
    filename = None

    while True:
        print("Input Filename: ")
        filename = input()

        if filename.endswith('.wav'):
            break
        else:
            print('Invalid Name! Must ends with .wav')

    audio.export(filename, format="wav")
    return audio

def adj_volume (audio, db):
    return audio + db

def adj_pitch (audio: AudioSegment, pitch):
    buffer = io.BytesIO()
    audio.export(buffer, format="wav")
    buffer.seek(0)

    shifted, sample_rate = librosa.load(buffer)
    shifted = librosa.effects.pitch_shift(shifted, sr=sample_rate, n_steps=pitch)

    buffer = io.BytesIO()
    sf.write(buffer, shifted, samplerate=sample_rate, format="wav")
    buffer.seek(0)

    return audio

def add_fadein (audio: AudioSegment):
    dur = random.randint(1,5) * 1000
    return audio.fade_in(dur)

def add_fadeout (audio: AudioSegment):
    dur = random.randint(1,5) * 1000
    return audio.fade_out(dur)

def add_whitenoise (audio: AudioSegment):
    noice = WhiteNoise().to_audio_segment(len(audio))
    noice = noice - 30
    return audio + noice

def reverse (audio: AudioSegment):
    return audio.reverse()

## Menu Function

In [5]:
def Menu ():
    audio = AudioSegment.from_wav(AUDIO_PATH)

    while True:
        print("1. Adjust Volume")
        print("2. Adjust Pitch")
        print("3. Insert Fade In")
        print("4. Insert Fade Out")
        print("5. Extra White Noise")
        print("6. Reverse Audio")
        print("0. Back to Main Menu")
        print(">> ")
        cc = input()

        if cc == "1":
            while True:
                print("Input Volume Change (-10) - (10000): ")
                val = input()

                if val < -10 or val > 10000:
                    print("Invalid Range")
                else:
                    break

            audio = adj_volume(audio, val)
            ExportMenu(audio)
        
        elif cc == "2":
            while True:
                print("Input Pitch (-10) - (10): ")
                val = input()

                if val < -10 or val > 10:
                    print("Invalid Range")
                else:
                    break

            audio = adj_volume(audio, val)
            ExportMenu(audio)
        
        elif cc == "3":
            audio = add_fadein(audio)
            ExportMenu(audio)

        elif cc == "4":
            audio = add_fadeout(audio)
            ExportMenu(audio)

        elif cc == "5":
            audio = add_whitenoise(audio)
            ExportMenu(audio)

        elif cc == "6":
            audio = reverse(audio)
            ExportMenu(audio)

        elif cc == "0":
            break

        else:
            print("Invalid Input")

In [6]:
def MainMenu ():
    while True:
        print("1. Audio Augmentation")
        print("2. Transcribe / Predict Text")
        print("0. Exit")
        print(">> ")
        cc = input()

        if cc == "1":
            Menu()

        elif cc == "2":
            text = Transcriber(AUDIO_PATH)
            print(text)

        elif cc == "0":
            break

        else:
            print("Invalid Input")

In [ ]:
MainMenu()

1. Audio Augmentation
2. Transcribe / Predict Text
0. Exit
>> 


ValueError: Expected the input Tensor to be 2D (batch, time). Found: [247680]